In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# ==========================================
# 1. GENERATE THE RAW MULTI-TENANT LOG FILE
# ==========================================
np.random.seed(42)
num_records = 1000
tenants = ['Acme_Corp', 'Beta_Industries', 'Delta_Systems', 'Omega_Labs']
tiers = ['Basic', 'Professional', 'Enterprise']
statuses = ['Active', 'Active', 'Active', 'Churned']

generated_data = []
start_date = datetime(2025, 1, 1)

for i in range(1, num_records + 1):
    tenant = np.random.choice(tenants)
    tier = np.random.choice(tiers)
    status = np.random.choice(statuses)
    
    monthly_price = 49.00 if tier == 'Basic' else (149.00 if tier == 'Professional' else 499.00)
    days_offset = np.random.randint(0, 500)
    signup_date = start_date + timedelta(days=days_offset)
    api_calls = np.random.randint(500, 100000) if status == 'Active' else np.random.randint(10, 5000)
    storage_used_gb = np.random.uniform(1.5, 500.0) if tier != 'Basic' else np.random.uniform(0.1, 50.0)
    
    generated_data.append({
        "account_id": f"ACC_{10000 + i}",
        "tenant_id": tenant,
        "subscription_tier": tier,
        "monthly_recurring_revenue": monthly_price,
        "account_status": status,
        "api_requests_count": api_calls,
        "cloud_storage_used_gb": round(storage_used_gb, 2),
        "activation_date": signup_date.strftime("%Y-%m-%d")
    })

raw_saas_df = pd.DataFrame(generated_data)
raw_csv_path = "raw_saas_tenant_logs.csv"
raw_saas_df.to_csv(raw_csv_path, index=False)
print(f"💾 File saved successfully as: '{raw_csv_path}'")

# ==========================================
# 2. INGEST DATA AND ROUTE SECURELY
# ==========================================
class TenantDataRouter:
    def __init__(self, file_path):
        self.data = pd.read_csv(file_path)
        
    def get_tenant_data(self, requesting_tenant_id):
        # Strict isolation filter
        isolated_data = self.data[self.data['tenant_id'] == requesting_tenant_id]
        
        # Security leak check
        leaked_rows = isolated_data[isolated_data['tenant_id'] != requesting_tenant_id]
        if not leaked_rows.empty:
            raise SecurityError("CRITICAL: Cross-Tenant Data Isolation Breach!")
            
        return isolated_data

# Initialize router with our file
router = TenantDataRouter(raw_csv_path)

# Test execution for Beta Industries
beta_isolated_data = router.get_tenant_data("Beta_Industries")
print(f"🔒 Beta_Industries isolation active. Row count fetched: {len(beta_isolated_data)}")

In [ ]:
import pandas as pd
import numpy as np

# 1. Load the Raw Asset File Generated Yesterday
raw_data_path = "raw_saas_tenant_logs.csv"
saas_pipeline_df = pd.read_csv(raw_data_path)

print("--- Initial Data Sanitation Inspection ---")
print(f"Total Rows Loaded: {saas_pipeline_df.shape[0]}")
print(f"Missing Values Check:\n{saas_pipeline_df.isnull().sum()}\n")

# 2. Simulate & Clean Data Pipeline Anomalies (Sanitation Phase)
# Let's ensure our activation date is correctly formatted as a datetime object
saas_pipeline_df['activation_date'] = pd.to_datetime(saas_pipeline_df['activation_date'])

# 3. Feature Engineering: Calculate Core SaaS Business Metrics
# A: Create a binary numeric flag for Churn Status (1 if Churned, 0 if Active)
saas_pipeline_df['is_churned'] = saas_pipeline_df['account_status'].apply(lambda x: 1 if x == 'Churned' else 0)

# B: Calculate Usage Intensity Indicator (API Requests per GB of Cloud Storage)
# Handles potential division by zero errors by adding a tiny epsilon value (1e-5)
saas_pipeline_df['usage_intensity'] = round(
    saas_pipeline_df['api_requests_count'] / (saas_pipeline_df['cloud_storage_used_gb'] + 1e-5), 2
)

# C: Segment Accounts based on High vs Low Platform Utilization Volume
storage_threshold = saas_pipeline_df['cloud_storage_used_gb'].median()
saas_pipeline_df['storage_tier_segment'] = saas_pipeline_df['cloud_storage_used_gb'].apply(
    lambda x: 'High_Utilization' if x > storage_threshold else 'Standard_Utilization'
)

# 4. Save the Refined/Cleaned Master File
cleaned_csv_path = "cleaned_saas_tenant_analytics.csv"
saas_pipeline_df.to_csv(cleaned_csv_path, index=False)

print("✅ Data Sanitation & Feature Engineering Pipeline Complete!")
print(f"💾 Processed analytical dataset saved as: '{cleaned_csv_path}'")
print("\n--- Processed Dataset Preview (Top 3 Rows) ---")
print(saas_pipeline_df[['account_id', 'tenant_id', 'is_churned', 'usage_intensity', 'storage_tier_segment']].head(3))

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime, timedelta

# 1. Force the notebook to look at your exact project directory
current_dir = os.getcwd()
print(f"Directory active: {current_dir}")

# 2. GENERATE RAW DATA
np.random.seed(42)
num_records = 1000
tenants = ['Acme_Corp', 'Beta_Industries', 'Delta_Systems', 'Omega_Labs']
tiers = ['Basic', 'Professional', 'Enterprise']
statuses = ['Active', 'Active', 'Active', 'Churned']

generated_data = []
start_date = datetime(2025, 1, 1)

for i in range(1, num_records + 1):
    tenant = np.random.choice(tenants)
    tier = np.random.choice(tiers)
    status = np.random.choice(statuses)
    
    monthly_price = 49.00 if tier == 'Basic' else (149.00 if tier == 'Professional' else 499.00)
    days_offset = np.random.randint(0, 500)
    signup_date = start_date + timedelta(days=days_offset)
    api_calls = np.random.randint(500, 100000) if status == 'Active' else np.random.randint(10, 5000)
    storage_used_gb = np.random.uniform(1.5, 500.0) if tier != 'Basic' else np.random.uniform(0.1, 50.0)
    
    generated_data.append({
        "account_id": f"ACC_{10000 + i}",
        "tenant_id": tenant,
        "subscription_tier": tier,
        "monthly_recurring_revenue": monthly_price,
        "account_status": status,
        "api_requests_count": api_calls,
        "cloud_storage_used_gb": round(storage_used_gb, 2),
        "activation_date": signup_date.strftime("%Y-%m-%d")
    })

# Save Raw File
raw_df = pd.DataFrame(generated_data)
raw_df.to_csv("raw_saas_tenant_logs.csv", index=False)
print("📦 Success: 'raw_saas_tenant_logs.csv' generated and saved!")

# 3. CLEAN & ENGINEER FEATURES
raw_df['activation_date'] = pd.to_datetime(raw_df['activation_date'])
raw_df['is_churned'] = raw_df['account_status'].apply(lambda x: 1 if x == 'Churned' else 0)
raw_df['usage_intensity'] = round(raw_df['api_requests_count'] / (raw_df['cloud_storage_used_gb'] + 1e-5), 2)

storage_threshold = raw_df['cloud_storage_used_gb'].median()
raw_df['storage_tier_segment'] = raw_df['cloud_storage_used_gb'].apply(
    lambda x: 'High_Utilization' if x > storage_threshold else 'Standard_Utilization'
)

# Save Cleaned File
raw_df.to_csv("cleaned_saas_tenant_analytics.csv", index=False)
print("✅ Success: 'cleaned_saas_tenant_analytics.csv' engineered and saved!")

In [1]:
import pandas as pd

# 1. Ingest the Cleaned Dataset Engineered on Day 2
cleaned_csv_path = "cleaned_saas_tenant_analytics.csv"
analytics_df = pd.read_csv(cleaned_csv_path)

print("--- Data Ingestion Verified ---")
print(f"Loaded {analytics_df.shape[0]} rows for performance aggregation.\n")

# 2. Build the Multi-Tenant Summary Aggregation Pipeline
# We group by 'tenant_id' to extract isolated operational footprints
tenant_summary = analytics_df.groupby('tenant_id').agg(
    total_accounts=('account_id', 'count'),
    total_mrr=('monthly_recurring_revenue', 'sum'),
    average_api_usage=('api_requests_count', 'mean'),
    total_storage_gb=('cloud_storage_used_gb', 'sum'),
    churned_accounts=('is_churned', 'sum')
).reset_index()

# 3. Compute Derived Executive KPIs per Tenant
# Calculate Churn Rate percentage for each corporate client
tenant_summary['churn_rate_pct'] = round(
    (tenant_summary['churned_accounts'] / tenant_summary['total_accounts']) * 100, 2
)

# Clean up averages formatting for presentation
tenant_summary['average_api_usage'] = round(tenant_summary['average_api_usage'], 0)
tenant_summary['total_mrr'] = round(tenant_summary['total_mrr'], 2)
tenant_summary['total_storage_gb'] = round(tenant_summary['total_storage_gb'], 2)

# 4. Export the Aggregated KPI Asset File
aggregated_csv_path = "tenant_executive_summary.csv"
tenant_summary.to_csv(aggregated_csv_path, index=False)

print("📈 Aggregation Pipeline Engine Executed Successfully!")
print(f"💾 Executive summary KPIs saved as: '{aggregated_csv_path}'")
print("\n--- Visualizing Final Day 3 Corporate Aggregation Grid ---")
print(tenant_summary)

--- Data Ingestion Verified ---
Loaded 1000 rows for performance aggregation.

📈 Aggregation Pipeline Engine Executed Successfully!
💾 Executive summary KPIs saved as: 'tenant_executive_summary.csv'

--- Visualizing Final Day 3 Corporate Aggregation Grid ---
         tenant_id  total_accounts  total_mrr  average_api_usage  \
0        Acme_Corp             260    59640.0            36958.0   
1  Beta_Industries             243    56907.0            37233.0   
2    Delta_Systems             264    56936.0            40184.0   
3       Omega_Labs             233    50767.0            37000.0   

   total_storage_gb  churned_accounts  churn_rate_pct  
0          41659.57                70           26.92  
1          43139.72                55           22.63  
2          44281.27                68           25.76  
3          36996.51                66           28.33  
